# 03 — Experiment Results

## Where results live

Each run appends to `outputs/ledger/<run>.jsonl` — one JSON line per node. A
sidecar `outputs/ledger/<run>.manifest.json` records the data-selection and
search-contract knobs (split seed, fractions, allowed levers, eta, …).

This notebook loads a ledger and lets you:
- Browse all nodes as a DataFrame
- Inspect any individual node (params, spec_config, scores)
- Visualise the search tree and (for search runs) the Ladder climb

**In the Setup cell, set `RUN` to a run name under `outputs/ledger/` (no
extension), e.g. `"2026-06-20-baseline"`.** The path is resolved against the
repo's `outputs/` dir, so it works from any working directory. If the run is
missing the cell lists the runs that *are* available and stops — it will not
fabricate data. Use `RUN = "demo"` for a synthetic example.

> Note: baseline/sweep runs are flat comparisons (no `val` queries), so the
> Ladder-climb and accepted/rejected views stay empty — those are for adaptive
> *search* runs.

In [ ]:
import os, warnings
import matplotlib.pyplot as plt
import pandas as pd

from scrye.config import OUTPUTS_DIR
from scrye.ledger import Ledger
from scrye.tree_view import tree_lines
from scrye.results import (
    nodes_frame, node_report, format_node_report,
    best_progression, spec_from_node, replay_proposals,
)
from scrye.search import GatePolicy, Proposal, SearchContract, run_search
from scrye.spec import PipelineSpec

# --- PARAMETERS (edit this) ---
RUN = "2026-06-20-baseline"   # a run name under outputs/ledger/ (no extension).
                              # Use RUN = "demo" for a synthetic example.
# -------------------------------

# Resolve against the repo's outputs/ dir (absolute) so this works no matter
# which directory the kernel runs in — Jupyter often runs from notebooks/, not
# the repo root, which is why a bare relative path silently missed the file.
LEDGER_DIR = OUTPUTS_DIR / "ledger"
RUN_PATH = str(LEDGER_DIR / f"{RUN}.jsonl")

if not os.path.exists(RUN_PATH):
    if RUN != "demo":
        # Do NOT fabricate a demo at a real run's path — fail loudly instead.
        avail = sorted(p.stem for p in LEDGER_DIR.glob("*.jsonl"))
        raise FileNotFoundError(
            f"No ledger at {RUN_PATH}.\n"
            f"Available runs in {LEDGER_DIR}: {avail or '(none)'}\n"
            f"Set RUN to one of those (no extension), or RUN = 'demo' for a synthetic example."
        )
    # RUN == 'demo': generate a tiny deterministic demo ledger. This branch only
    # runs when you explicitly ask for the demo, so it never masks a real run.
    print("[DEMO] Generating a synthetic demo ledger with a stub score_fn.\n")
    LEDGER_DIR.mkdir(parents=True, exist_ok=True)
    _proposals_list = [
        Proposal("recalib.global_temp", {"T": 1.5}, "Flatten over-confident instruct distributions"),
        Proposal("recalib.dirichlet", {"alpha": 0.05}, "Add smoothing toward uniform to lift under-predicted tails"),
    ]
    _dev_scores = [42.3, 45.1]
    _val_scores = [43.8, 46.5]
    _call_counts = {"dev": 0, "val": 0}

    def _stub_score_fn(spec, recs):
        if recs == ["val"]:
            i = min(_call_counts["val"], len(_val_scores) - 1)
            _call_counts["val"] += 1
            return _val_scores[i], {"cost_usd": 0.001}
        i = min(_call_counts["dev"], len(_dev_scores) - 1)
        _call_counts["dev"] += 1
        return _dev_scores[i], {"cost_usd": 0.002}

    _it = iter(_proposals_list)
    _demo_ledger = Ledger(RUN_PATH)
    _contract = SearchContract(
        goal="Improve SimBench S score on dev+val",
        allowed_levers=["recalib.global_temp", "recalib.dirichlet"],
        autonomy_budget=10,
        val_query_budget=5,
        gate_policy=GatePolicy(gate_on_surprise=False),
        eta=0.5,
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        run_search(
            _contract, _demo_ledger,
            dev=["dev"], val=["val"],
            score_fn=_stub_score_fn,
            propose_fn=lambda state: next(_it, None),
            now_fn=lambda: "2026-06-20T00:00:00",
        )
    from scrye.manifest import RunManifest, dataset_fingerprint
    from scrye.levers import REGISTRY_VERSION
    RunManifest(
        registry_version=REGISTRY_VERSION, split_seed=0,
        split_fractions={"dev": 0.5, "val": 0.25, "test": 0.25}, split_unit="question",
        pin_required_to="test", goal=_contract.goal, allowed_levers=_contract.allowed_levers,
        autonomy_budget=_contract.autonomy_budget, val_query_budget=_contract.val_query_budget,
        eta=_contract.eta, cost_cap_usd=1000.0, model="gemini-flash-lite",
        temperature=0.0, seed=0, dataset_fingerprint=dataset_fingerprint([]),
        created_at="2026-06-20T00:00:00",
    ).save(RUN_PATH.replace(".jsonl", ".manifest.json"))
    print(f"[DEMO] Wrote {RUN_PATH} ({len(_demo_ledger.nodes)} nodes) + manifest.\n")

print(f"Using ledger: {RUN_PATH}")


In [ ]:
ledger = Ledger.load(RUN_PATH)
print(f"Loaded {len(ledger.nodes)} nodes, global_k={ledger.global_k()}, "
      f"best_val={ledger.best_val():.4f}, total_cost=${ledger.total_cost():.4f}\n")
print("Search tree:")
print("\n".join(tree_lines(ledger)))


In [ ]:
df = nodes_frame(ledger)
display(df)

# Scatter: dev_score vs val_score, coloured by accepted status
val_rows = df[df["val_score"].notna()].copy()
if not val_rows.empty:
    colors = val_rows["accepted"].map({True: "tab:green", False: "tab:red", None: "tab:gray"})
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(val_rows["dev_score"], val_rows["val_score"], c=colors, s=80, edgecolors="k", linewidths=0.5)
    ax.set_xlabel("Dev score")
    ax.set_ylabel("Val score")
    ax.set_title("Dev vs Val score (green=accepted, red=rejected)")
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor="tab:green", label="accepted"),
        Patch(facecolor="tab:red", label="rejected"),
        Patch(facecolor="tab:gray", label="no val query"),
    ]
    ax.legend(handles=legend_elements)
    plt.tight_layout()
    plt.show()
else:
    print("No val-queried nodes to plot.")


In [ ]:
# Change NODE_ID to inspect a different node
NODE_ID = ledger.nodes[-1].node_id  # last node by default

print(format_node_report(ledger, NODE_ID))


In [ ]:
bp = best_progression(ledger)
display(bp)

if not bp.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(bp["step"], bp["best_so_far"], marker="o", color="steelblue", label="best_so_far")
    ax.scatter(bp["step"], bp["val_score"], color="lightgray", zorder=3, label="val_score (all)")
    # Mark new-best points
    new_best = bp[bp["is_new_best"]]
    ax.scatter(new_best["step"], new_best["best_so_far"], color="gold",
               s=120, zorder=4, edgecolors="darkorange", linewidths=1.5, label="new best")
    ax.set_xlabel("Step (val query index)")
    ax.set_ylabel("Val score")
    ax.set_title("Ladder climb — best val score over search steps")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No val-queried nodes for progression plot.")


## Reproducibility

### How to reproduce this run

The sidecar `<run>.manifest.json` records every knob that was deterministic at search time:
split seed/fractions/unit, allowed levers, eta, model, temperature, and a dataset fingerprint.

To reproduce:

1. **Re-select the data** — call `make_split(records, seed=manifest.split_seed, fractions=manifest.split_fractions, unit=manifest.split_unit, pin_required_to=manifest.pin_required_to)` to get identical dev/val/test sets.
2. **Replay the proposals** — `replay_proposals(ledger)` returns the exact sequence of `Proposal(lever_id, params, rationale)` objects recorded in the ledger.
3. **Re-run search** — pass a fresh `Ledger`, the same `SearchContract`, and a `propose_fn` that exhausts the replayed list.

**Why it's identical:** cached LLM calls (temp=0 + on-disk cache) are byte-for-byte repeatable. Uncached calls with temp=0 are near-identical but not guaranteed by the API. The split and proposal sequence are pure functions of the recorded seeds.

The cell below runs an **offline reproducibility check** using the same stub score function and asserts that the replayed ledger matches the original.

In [ ]:
# Load manifest if present
import json as _json
manifest_path = RUN_PATH.replace(".jsonl", ".manifest.json")
if os.path.exists(manifest_path):
    from scrye.manifest import RunManifest
    print("Run manifest:")
    for k, v in _json.loads(open(manifest_path).read()).items():
        print(f"  {k}: {v}")
else:
    print("(No manifest file found.)")

print()

# The offline replay check below uses a deterministic STUB scorer, so it only
# applies to the synthetic 'demo' run. For real runs, reproducibility comes from
# the on-disk LLM cache (temp=0) plus the seeded split/proposal sequence — see
# the manifest above and `replay_proposals(ledger)`.
if RUN != "demo":
    print(f"\nOffline replay check is demo-only; skipping for real run '{RUN}'.")
    print("Reproducibility for real runs: cached temp=0 LLM calls + seeded split/proposals.")
else:
    _replayed_proposals = replay_proposals(ledger)
    _replay_iter = iter(_replayed_proposals)
    _rc = {"dev": 0, "val": 0}
    _dscores = [42.3, 45.1]
    _vscores = [43.8, 46.5]

    def _replay_score_fn(spec, recs):
        if recs == ["val"]:
            i = min(_rc["val"], len(_vscores) - 1)
            _rc["val"] += 1
            return _vscores[i], {"cost_usd": 0.001}
        i = min(_rc["dev"], len(_dscores) - 1)
        _rc["dev"] += 1
        return _dscores[i], {"cost_usd": 0.002}

    import tempfile as _tmp
    _replay_ledger = Ledger(os.path.join(_tmp.mkdtemp(), "replay.jsonl"))
    _replay_contract = SearchContract(
        goal="Improve SimBench S score on dev+val",
        allowed_levers=["recalib.global_temp", "recalib.dirichlet"],
        autonomy_budget=10,
        val_query_budget=5,
        gate_policy=GatePolicy(gate_on_surprise=False),
        eta=0.5,
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        run_search(
            _replay_contract, _replay_ledger,
            dev=["dev"], val=["val"],
            score_fn=_replay_score_fn,
            propose_fn=lambda state: next(_replay_iter, None),
            now_fn=lambda: "2026-06-20T00:00:00",
        )

    orig_levers = [n.lever_id for n in ledger.nodes]
    replay_levers = [n.lever_id for n in _replay_ledger.nodes]
    assert orig_levers == replay_levers, f"Lever mismatch: {orig_levers} != {replay_levers}"
    assert ledger.global_k() == _replay_ledger.global_k(), (
        f"global_k mismatch: {ledger.global_k()} != {_replay_ledger.global_k()}"
    )
    print(f"Original  : levers={orig_levers}, global_k={ledger.global_k()}")
    print(f"Replayed  : levers={replay_levers}, global_k={_replay_ledger.global_k()}")
    print("REPRODUCED ✓")


---
## Cross-experiment analysis

The plots below read the saved runs in `outputs/runs/` and show the **grouped**
SimBench score — the demographic-conditioning task, where method differences
actually show. (Pop is mostly unconditioned, so methods bunch there.)


In [ ]:
import io, json
import numpy as np
from matplotlib.figure import Figure
from scrye import evaluate as ev
from scrye import viz

RUNS_DIR = OUTPUTS_DIR / "runs"

# Render returned Figure objects inline (same trick as 00_starter).
def _fig_png(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    return buf.getvalue()
get_ipython().display_formatter.formatters["image/png"].for_type(Figure, _fig_png)

def grouped_table(run):
    """Per-system grouped SimBench score (mean + 95% CI) from a saved run."""
    per = json.load(open(RUNS_DIR / f"{run}.results.json"))
    rows = []
    for key, recs in per.items():
        sb = ev.score_by(pd.DataFrame(recs), "split")
        g = sb[sb["split"] == "grouped"]
        if len(g):
            rows.append({"key": key,
                         "mean_score": float(g.iloc[0]["mean_score"]),
                         "ci_low": float(g.iloc[0]["ci_low"]),
                         "ci_high": float(g.iloc[0]["ci_high"])})
    return pd.DataFrame(rows)


### Stage 01 — conditioning strategies (gemini-2.5-flash-lite)

In [ ]:
t1 = grouped_table("2026-06-20-baseline").rename(columns={"key": "system"})
display(t1.sort_values("mean_score", ascending=False).round(2).reset_index(drop=True))
f = viz.plot_system_comparison(t1, label_col="system")
f.axes[0].set_title("Stage 01 — conditioning strategy (grouped SimBench S)")
f


### Stage 02 — model portability (top strategies × models)

In [ ]:
t2 = grouped_table("2026-06-20-model-sweep")
t2[["model", "system"]] = t2["key"].str.split("::", expand=True)
order_m = ["gemini-flash-lite", "gemini-3.1-flash-lite", "gemini-3.5-flash"]
piv = t2.pivot(index="system", columns="model", values="mean_score")
piv = piv[[m for m in order_m if m in piv.columns]]
display(piv.round(2))

fig, ax = plt.subplots(figsize=(8, 4))
systems = piv.index.tolist(); x = np.arange(len(systems)); w = 0.25
for i, m in enumerate(piv.columns):
    ax.bar(x + (i - 1) * w, piv[m].values, w, label=m.replace("gemini-", ""))
ax.set_xticks(x); ax.set_xticklabels(systems)
ax.set_ylabel("grouped SimBench S"); ax.axhline(0, color="0.6", ls="--", lw=1)
ax.set_title("Stage 02 — model portability (grouped)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


### Stage 03 — persona mechanisms (gemini-3.1-flash-lite)

In [ ]:
t3 = grouped_table("2026-06-20-persona-mechanisms").rename(columns={"key": "system"})
display(t3.sort_values("mean_score", ascending=False).round(2).reset_index(drop=True))
f = viz.plot_system_comparison(t3, label_col="system")
f.axes[0].set_title("Stage 03 — persona mechanism (grouped SimBench S)")
f


### Does the original *faithful* baseline benefit from a better model?

**Yes.** This is the `faithful` row of the Stage 02 sweep — the original SimBench
first-person persona, held fixed, across model generations.


In [ ]:
fa = t2[t2["system"] == "faithful"].set_index("model").reindex(order_m)
display(fa[["mean_score", "ci_low", "ci_high"]].round(2))

fig, ax = plt.subplots(figsize=(7, 4))
xm = np.arange(len(fa)); means = fa["mean_score"].to_numpy()
err = np.vstack([means - fa["ci_low"].to_numpy(), fa["ci_high"].to_numpy() - means])
ax.bar(xm, means, 0.6, yerr=err, capsize=4, color="#C44E52", alpha=0.85)
ax.set_xticks(xm); ax.set_xticklabels([m.replace("gemini-", "") for m in fa.index])
ax.set_ylabel("grouped SimBench S"); ax.axhline(0, color="0.6", ls="--", lw=1)
ax.set_title("Original 'faithful' first-person baseline, across models")
for xi, m in zip(xm, means):
    ax.annotate(f"{m:.1f}", (xi, m), ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()
print("The naive first-person baseline nearly doubles its grouped score just by "
      "moving to gemini-3.1-flash-lite — the better model recovers most of the gap.")


---
## Final system — full results

Carry-forward winner: **`contextualized`** conditioning on
**`gemini-3.1-flash-lite`**. Same analysis suite as `00_starter`. The cell below
runs it on a seeded, dataset-stratified sample (live LLM calls — mostly cached,
so cheap).


In [ ]:
from scrye.data import load_split
from scrye.experiment import build_pipeline

pop = load_split("pop"); grouped = load_split("grouped")
normalizers = ev.build_normalizers(pop + grouped)

winner = build_pipeline(model="gemini-3.1-flash-lite", predictor="contextualized")
sample = ev.stratified_sample(pop + grouped, n=300, seed=0)
final = ev.evaluate(winner, sample, normalizers=normalizers, max_workers=6)

s = ev.summarize(final)
print(f"FINAL SYSTEM: {winner.name} on gemini-3.1-flash-lite")
print(f"mean SimBench S    : {s['mean_score']:.1f}  (95% CI {s['ci_low']:.1f}, {s['ci_high']:.1f})")
print(f"frac below uniform : {s['frac_below_uniform']:.1%}    n = {s['n']}")


In [ ]:
from IPython.display import display

display(ev.score_by(final, "dataset").round(1))
for fig in [viz.plot_score_histogram(final),
            viz.plot_score_by_dataset(final),
            viz.plot_score_vs_entropy(final),
            viz.plot_calibration(final),
            viz.plot_example_distributions(final, n=6)]:
    display(fig)


---
## Ask the winning system a new question + demographic

`ask_winner(question, options, segment=..., year=...)` runs the winning
prompting strategy (`contextualized` on `gemini-3.1-flash-lite`) on a brand-new
item and returns its predicted answer distribution. One live LLM call.


In [ ]:
from scrye.data import SimBenchRecord
from scrye.experiment import build_pipeline

def ask_winner(question, options, segment=None, year=None,
               model="gemini-3.1-flash-lite", strategy="contextualized", show=True):
    """Predict an answer distribution for a NEW question + demographic.

    options : list of "A) text" strings, or bare labels.
    segment : dict, e.g. {"country": "Finland", "age_group": "30-49", "gender": "female"}.
    """
    seg = dict(segment or {})
    if options and ")" in str(options[0]):
        labels = [str(o).split(")")[0].strip() for o in options]
        body = "\n".join(str(o) for o in options)
    else:
        labels = [str(o) for o in options]
        body = "\n".join(f"{l}) {l}" for l in labels)
    gp = []
    if year:
        gp.append(f"The year is {year}.")
    country = seg.get("country") or seg.get("cntry")
    if country:
        gp.append(f"You are from {country}.")
    rec = SimBenchRecord("ad_hoc", "grouped", f"{question.strip()}\n{body}",
                         tuple(labels), {l: 1 / len(labels) for l in labels},
                         " ".join(gp), segment=seg, num_grouping_vars=len(seg))
    dist = build_pipeline(model=model, predictor=strategy).predict(rec)
    if show:
        fig, ax = plt.subplots(figsize=(6, 0.5 * len(labels) + 1.2))
        ax.barh(list(dist), list(dist.values()), color="#55A868")
        ax.invert_yaxis(); ax.set_xlim(0, 1); ax.set_xlabel("predicted probability")
        ax.set_title(f"{strategy} @ {model}\n{seg or 'general population'}")
        for l, v in dist.items():
            ax.annotate(f"{v:.2f}", (v, l), xytext=(3, 0),
                        textcoords="offset points", va="center", fontsize=9)
        plt.tight_layout(); plt.show()
    return dist


# Example — edit freely:
_ = ask_winner(
    "How important is it for you to live in a democratically governed country?",
    ["A) Not important", "B) Somewhat important", "C) Very important"],
    segment={"country": "Finland", "age_group": "65+"}, year=2024,
)
